# fpm-rs Python quickstart
Configure a microscope, simulate NumPy data, and reconstruct it with the Rust core.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import fpm_rs as fpm

In [ ]:
optics = fpm.Optics(532e-9, 0.10, 4.0, 6.5e-6)
leds = fpm.LEDArray((3, 3), 4e-3, 90e-3, (1.0, 1.0))
model = fpm.compile_model(optics, leds, (32, 32), (64, 64))
row, column = np.indices(model.reconstruction_shape)
true_amplitude = 0.7 + 0.3 * ((row // 8 + column // 8) % 2)
true_phase = 0.4 * np.sin(row / 7.0) * np.cos(column / 9.0)
object_field = np.asarray(true_amplitude * np.exp(1j * true_phase), dtype=np.complex128)

In [ ]:
simulation = fpm.simulate(model, object_field, seed=1234)
problem = fpm.ReconstructionProblem(simulation.measurements, simulation.reconstruction_model)
result = fpm.AlternatingProjection(iterations=20).run(problem)

In [ ]:
truth = simulation.ground_truth_object
figure, axes = fpm.plot.plot_reconstruction(truth, result)
display(figure)
# figure.savefig("quickstart_reconstruction.png", dpi=180, bbox_inches="tight")
plt.close(figure)